# Project 1 — COVID-19 Vaccination Statistical Analysis
## Exploratory Data Analysis and Computational Extension

**Original research project:** Application of the Kruskal-Wallis Test to COVID-19 Vaccination Data — A Case Study at UMaT.

This notebook documents a 2026 computational extension of the undergraduate project. It focuses on data quality, cleaning, descriptive statistics, exploratory analysis, and relationships that can be examined using the variables present in the available response export.

> **Important:** The available response export does not contain the academic-year variable used in the original 2023 Kruskal-Wallis analysis. This notebook therefore does **not** claim to reproduce the original Kruskal-Wallis result.

## Data privacy

The respondent-level source data should remain private and should **not** be committed to a public GitHub repository without appropriate permission.

Place the private CSV in a local `data/` directory when running this notebook. Public GitHub outputs should be aggregated tables, figures, and documentation only.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency

# Local/private source file. Do not commit this file to a public repository.
df = pd.read_csv("../data/project_dataset_cleaned_PRIVATE.csv")
df.head()

## 1. Dataset structure

In [ ]:
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("\nColumns:")
print(df.columns.tolist())
print("\nDuplicate rows:", df.duplicated().sum())
df.info()

## 2. Missing and invalid values

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0]

In [ ]:
# The original dataset contains one non-numeric age response ("wo").
pd.to_numeric(df["Age"], errors="coerce").describe()

## 3. Demographic summary

In [ ]:
print("Gender:")
display(df["Sex/Gender"].value_counts(dropna=False))

print("\nValid age summary:")
display(df["Age_clean"].describe())

In [ ]:
age_counts = df["Age_clean"].dropna().value_counts().sort_index()
age_counts.plot(kind="bar", figsize=(7,5))
plt.title("Age Distribution (Valid Numeric Ages)")
plt.xlabel("Age")
plt.ylabel("Number of respondents")
plt.tight_layout()
plt.show()

In [ ]:
gender_counts = df["Sex/Gender"].value_counts()
gender_counts.plot(kind="bar", figsize=(7,5))
plt.title("Gender Distribution")
plt.xlabel("Sex/Gender")
plt.ylabel("Number of respondents")
plt.tight_layout()
plt.show()

## 4. Vaccination patterns

In [ ]:
print("Vaccine type:")
display(df["Vaccine Type"].value_counts(dropna=False))

print("\nDose:")
display(df["Dose"].value_counts(dropna=False))

In [ ]:
vaccine_counts = df["Vaccine Type"].value_counts()
vaccine_counts.plot(kind="bar", figsize=(8,5))
plt.title("Reported Vaccine Type")
plt.xlabel("Vaccine type")
plt.ylabel("Number of respondents")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
dose_counts = df["Dose"].value_counts()
dose_counts.plot(kind="bar", figsize=(8,5))
plt.title("Reported Dose Status")
plt.xlabel("Dose")
plt.ylabel("Number of respondents")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 5. Vaccination status

In [ ]:
vaccinated_types = [
    "Pfizer/BioNTech",
    "Moderna Spikevax",
    "Janssen(Johnson and Johnson)",
    "Astrazeneca/oxford formulation (2)",
    "Astrazeneca/oxford vaxzevrial(1)",
    "Serum institute of India",
]
not_vaccinated_types = ["Haven't had a vaccine yet", "None"]

status = np.where(
    df["Vaccine Type"].isin(vaccinated_types), "Vaccinated",
    np.where(df["Vaccine Type"].isin(not_vaccinated_types),
             "Not vaccinated", pd.NA)
)

df["Vaccination_status"] = status
display(df["Vaccination_status"].value_counts(dropna=False))

### Interpretation

Vaccination status is derived from the reported vaccine-type field. Missing vaccine-type responses remain unknown rather than being classified as vaccinated or unvaccinated.

This distinction is important because a missing response is not evidence of either vaccination status.

## 6. Gender and vaccination status

In [ ]:
mask = df["Sex/Gender"].notna() & df["Vaccination_status"].notna()
ct = pd.crosstab(
    df.loc[mask, "Sex/Gender"],
    df.loc[mask, "Vaccination_status"]
)
display(ct)

chi2, p, dof, expected = chi2_contingency(ct)
print(f"Chi-square statistic: {chi2:.3f}")
print(f"Degrees of freedom: {dof}")
print(f"p-value: {p:.3f}")

print("\nNote: the very small number of unvaccinated observations means this test should be interpreted cautiously.")

## 7. Limitations

1. The available export contains substantial missingness in age.
2. One age response is non-numeric (`wo`).
3. Several vaccine-type responses are missing.
4. The academic-year variable used in the original Kruskal-Wallis analysis is not present in this export.
5. The survey is an undergraduate project dataset and should not be interpreted as a representative sample of all university students.
6. Any inferential test in this extension should be interpreted in light of the small number of observations in some categories.

## 8. Conclusion

This computational extension demonstrates a reproducible workflow for inspecting, cleaning, summarizing, and exploring a real survey dataset.

The analysis deliberately distinguishes the original 2023 statistical research from the 2026 computational extension. This prevents unsupported claims about reproducing the original Kruskal-Wallis analysis.